# 16 · Choosing the Right Collection

**Level 2 · Data Structures and Functions** &nbsp;|&nbsp; Previous: [15_Sets](15_Sets.ipynb) &nbsp;|&nbsp; Next: [17_Functions](17_Functions.ipynb)

You now know four collections: lists, tuples, dictionaries and sets (plus strings, which behave like collections of characters). This notebook is a **consolidation**: it puts them side by side so you can decide, for any problem, which one to reach for.

### 🎯 Learning goals

By the end of this notebook you will be able to:

- compare lists, tuples, dictionaries, sets and strings on **order, duplicates, lookup and mutability**
- explain which values are **mutable** and which are **immutable**, and what that means in practice
- say what can be used as a **dictionary key** (or a set item) and what cannot, and why
- combine collections: a **list of dicts**, a **dict of lists**, a **dict of dicts**
- pick the right structure for a described scenario and model a small **gradebook**

In [ ]:
import sys

sys.path.insert(0, "../src")
from course_tools import check, check_output

## 1 · Why this matters

Almost every program is about data, and the first design decision is: *what shape should the data have?* The same information can be stored in a list, a dictionary or a set, but the choice changes how easy the rest of the program is to write.

- Looking up a phone number by name in a **list** means searching every entry. In a **dict** it is one step.
- Removing duplicates from a **list** takes a loop. A **set** does it by being a set.
- Storing a point on a map as a **list** lets anyone change it by accident. A **tuple** doesn't.

Choosing well makes code shorter, faster and harder to get wrong. Choosing badly makes you fight the data. The skill you are practising here is not new syntax; it is **judgement**.

## 2 · The five collections side by side

Here are the same few colours in each shape:

In [ ]:
as_list = ["red", "green", "blue", "green"]
as_tuple = ("red", "green", "blue", "green")
as_dict = {"red": "#ff0000", "green": "#00ff00", "blue": "#0000ff"}
as_set = {"red", "green", "blue", "green"}
as_str = "red green blue green"

print(as_list)
print(as_tuple)
print(as_dict)
print(sorted(as_set), "<- a set, shown sorted because a set has no order")
print(as_str)

Notice what each one did with the repeated `"green"`: the list and tuple kept both, the string kept both words, the set quietly kept one, and the dict can't even *express* a repeat (every key is unique; we had to attach a value instead).

### The comparison table

| | **list** | **tuple** | **dict** | **set** | **str** |
|---|---|---|---|---|---|
| Written as | `[1, 2]` | `(1, 2)` | `{"a": 1}` | `{1, 2}` | `"hi"` |
| Holds | any items | any items | key → value pairs | unique items | characters |
| Ordered? | yes | yes | remembers insertion order | **no** | yes |
| Index with `x[0]`? | yes | yes | no (use a **key**: `x["a"]`) | **no** | yes |
| Duplicates? | allowed | allowed | keys unique, values can repeat | **never** | allowed |
| Mutable? | **yes** | no | **yes** | **yes** | no |
| Loop with `for` gives | items | items | keys | items | characters |
| `in` asks | "is this item present?" | "is this item present?" | "is this **key** present?" | "is this item present?" | "is this text present?" |

Every one of them works with `len(x)`, `for ... in x` and `in`:

In [ ]:
collections = [
    ("list", as_list),
    ("tuple", as_tuple),
    ("dict", as_dict),
    ("set", as_set),
    ("str", as_str),
]

for name, thing in collections:
    print(f"{name:5} len={len(thing):2}  'green' in it: {'green' in thing}")

Two things to notice. `len(as_dict)` counts **keys** (3, not 6), and `"green" in as_dict` checks the keys. And `"green" in as_str` is a *substring* test: it would also be `True` for `"gre"`.

### Same facts, different shapes

Suppose we know three facts about a person. Which collection fits?

In [ ]:
person_as_list = ["Ada", 36, "London"]
person_as_dict = {"name": "Ada", "age": 36, "city": "London"}

print(person_as_list[2])        # what was position 2 again?
print(person_as_dict["city"])   # the label says what it is

A useful rule of thumb:

- **Many things of the same kind** (a list of scores, a list of names): use a **list** (or a tuple if it should never change).
- **A few different facts about one thing**, accessed by name: use a **dict**.
- **A fixed bundle where position carries the meaning** (an `(x, y)` point, an `(r, g, b)` colour): use a **tuple**.
- **A pile where only membership matters** and duplicates are noise: use a **set**.

### Try it

Take the sentence `"to be or not to be"`. Create, from the words of that sentence: a list of the words (`.split()`), then a set of the unique words, and print the length of each. Which is smaller, and why?

In [ ]:
# your code here

## 3 · Mutable versus immutable

A **mutable** object can be changed in place after it is created. An **immutable** object never changes; any "change" actually builds a new object.

| Mutable | Immutable |
|---|---|
| list, dict, set | tuple, str, and the numbers/booleans you met in Level 1 (`int`, `float`, `bool`) |

Mutable collections have methods like `append`, `remove`, `add` and `update` that change them in place:

In [ ]:
shopping = ["milk", "eggs"]
shopping.append("bread")        # the SAME list, now longer
print(shopping)

stock = {"apples": 4}
stock["pears"] = 7              # the SAME dict, now bigger
print(stock)

seen = {1, 2}
seen.add(3)                     # the SAME set, now bigger
print(seen)

Immutable ones refuse. Trying to change a tuple or a string in place is an error, shown on purpose below. ⚠️ **Expected to fail.**

In [ ]:
point = (3, 4)
point[0] = 10

In [ ]:
word = "cat"
word[0] = "b"

With an immutable value you make a *new* one instead:

In [ ]:
point = (10, 4)          # a new tuple replaces the old one
word = "b" + word[1:]    # a new string, built from pieces
print(point, word)

### Why does it matter which you pick?

- **Safety.** If a value should never change (days of the week, the corners of a board), an immutable type *enforces* that.
- **Sharing.** Two names for the same mutable object are an **alias** (notebook 12): change it through one name and the other sees it. Immutable objects can be shared freely because nobody can change them.
- **Dictionary keys and set items.** Only certain values are allowed there. That is the next section.

A reminder of the alias gotcha, because it applies to every mutable collection, not just lists:

In [ ]:
original = {"a": 1}
alias = original            # NOT a copy: two names, one dict
alias["b"] = 2
print(original)

independent = original.copy()   # a real (outer) copy
independent["c"] = 3
print(original, independent)

### Try it

Without running it first, decide which of these lines would raise an error, then run them one at a time to check: `(1, 2, 3).append(4)`, `"hello".upper()`, `{1, 2}.add(3)`, `"hello"[0]`. Careful: some are fine!

In [ ]:
# your code here (one line at a time)

## 4 · What can be a dictionary key?

A dictionary finds a value from its key very quickly, almost as if it jumps straight to the right spot. For that trick to work, a key must **never change while it is in the dictionary**: if it could change, the dictionary would be looking in the wrong spot. (Python's name for "can be used this way" is **hashable**; you do not need the mechanics, just the rule.)

> **Rule of thumb.** Immutable values can be keys. Mutable ones cannot.

| Can be a key | Cannot be a key |
|---|---|
| `str`, `int`, `float`, `bool` | `list` |
| `tuple` (if everything inside it is also immutable) | `dict` |
| | `set` |
| | a `tuple` that contains a list |

In [ ]:
prices = {
    "apple": 0.5,          # str key
    7: "lucky",            # int key
    (2, 3): "treasure",    # tuple key
}
print(prices["apple"], prices[7], prices[(2, 3)])

Tuples as keys are very handy: a **position on a grid** or a **(first, last) name pair** is one value made from two parts.

In [ ]:
board = {}
board[(0, 0)] = "X"
board[(1, 1)] = "O"
print(board)
print((1, 1) in board, (2, 2) in board)

A list cannot be a key. ⚠️ **Expected to fail.** The error is called `unhashable type`, Python's way of saying "this is mutable, so I can't use it as a key".

In [ ]:
bad = {[0, 0]: "X"}

The same rule applies to **set items**, because sets use the same quick-lookup trick:

In [ ]:
ok = {(0, 0), (1, 1)}      # tuples are fine
print(len(ok))

In [ ]:
bad_set = {[0, 0], [1, 1]}

**Values** have no such limits. A dict value can be anything, including a list, a set or another dict. That is the starting point for the next section.

### Try it

Make a dictionary called `visits` whose keys are `(city, year)` tuples, with the value being a visitor count. Add `("Paris", 2023)` with `1200` and `("Paris", 2024)` with `1350`, then print the count for `("Paris", 2024)`.

In [ ]:
# your code here

## 5 · A decision guide

Ask these questions, in this order:

1. **Do I need to look things up by a name or label?** Use a **dict**. (Keys give you the label, values the data.)
2. **Do I only care whether something is present, and not about duplicates or order?** Use a **set**.
3. **Does the order matter, or may items repeat?** Then it is a sequence:
    - will it grow, shrink or be edited? Use a **list**.
    - should it stay fixed (a record, a coordinate, something you want to use as a key)? Use a **tuple**.
4. **Is it just text?** Use a **string**.

### Speed, as intuition only

Asking `x in some_list` makes Python check the items **one by one** until it finds `x` or runs out. With a million items that is a lot of checking. Asking `x in some_set` or `x in some_dict` goes almost straight to the right place, whatever the size. So if your program does *many* membership tests or lookups on a big collection, a set or dict pays off. For a handful of items it makes no practical difference; choose for clarity first.

### Scenarios

Look at how the requirements point to a structure:

In [ ]:
# "Which of these usernames are taken?" -> membership only -> set
taken = {"ada", "grace", "linus"}
print("ada" in taken, "alan" in taken)

# "The steps of a recipe, in order, and I'll add more" -> list
steps = ["chop", "fry"]
steps.append("serve")
print(steps)

# "Convert a country code to a country name" -> lookup by label -> dict
countries = {"FR": "France", "JP": "Japan"}
print(countries["JP"])

# "A point on a map" -> fixed, position-meaning -> tuple
location = (51.5, -0.12)
print(location)

### Converting between collections

You can change shape with the names of the types, which makes a **new** collection from the items of the old one:

In [ ]:
words = ["b", "a", "b", "c"]

print(set(words))              # duplicates dropped (order not guaranteed)
print(sorted(set(words)))      # unique AND in order, as a list
print(tuple(words))            # frozen copy
print(list("abc"))             # a string's characters, as a list
print(dict(zip(["x", "y"], [1, 2])))   # two lists -> one dict

Note that `sorted()` always gives back a **list**, whatever you hand it. And converting to a set loses the order and the duplicates; you cannot get them back.

### Try it

You are given `names = ["Cy", "Ana", "Cy", "Ben", "Ana"]`. In one line each, print: the number of *distinct* names, and the distinct names in alphabetical order.

In [ ]:
names = ["Cy", "Ana", "Cy", "Ben", "Ana"]
# your code here

## 6 · Mixed structures

Real data is rarely a flat list of numbers. The collections nest: any item or value can itself be a collection.

### A list of dicts: many records with the same fields

This is the shape of a table: each dict is a row, each key a column name.

In [ ]:
students = [
    {"name": "Ana", "grade": 90},
    {"name": "Ben", "grade": 72},
    {"name": "Cy", "grade": 85},
]

print(students[1])              # one row
print(students[1]["name"])      # one cell: row 1, column "name"

for student in students:
    print(student["name"], "scored", student["grade"])

Read `students[1]["name"]` left to right: *take the list item at position 1 (a dict), then look up `"name"` in it.* Each pair of brackets peels off one layer.

Searching and totalling work with the loop patterns of Level 1:

In [ ]:
total = 0
passed = []
for student in students:
    total += student["grade"]
    if student["grade"] >= 80:
        passed.append(student["name"])

print("average:", total / len(students))
print("80 or above:", passed)

**Fits when:** you have many similar records and you go through them in order. **Awkward when:** you often need *one particular* record by name, since you must loop to find it.

### A dict of lists: several values under one label

Each key owns a list that can grow.

In [ ]:
courses = {
    "math": ["Ana", "Ben"],
    "art": ["Cy"],
}

courses["art"].append("Ana")        # add to one key's list
courses["music"] = ["Ben"]          # new key, new list

for course in courses:
    print(course, "->", courses[course])

**Fits when:** you group items by a category (students by course, words by first letter). Notebook 14's grouping exercise builds exactly this.

### A dict of dicts: records looked up by name

Make the name the key, and put the other fields in an inner dict. Now finding a record is one lookup:

In [ ]:
profiles = {
    "ada": {"age": 36, "city": "London"},
    "grace": {"age": 45, "city": "New York"},
}

print(profiles["grace"]["city"])
profiles["ada"]["age"] = 37             # change one inner value
profiles["linus"] = {"age": 28, "city": "Helsinki"}   # add a whole record
print(profiles["ada"])
print(len(profiles))

### Other combinations you will meet

| Shape | Example | Typical use |
|---|---|---|
| list of lists | `[[1, 2], [3, 4]]` | grid / matrix (notebook 12) |
| list of tuples | `[("Ana", 90), ("Ben", 72)]` | fixed pairs, easy to loop with unpacking |
| dict with tuple keys | `{(0, 0): "X"}` | grids, coordinates, composite keys |
| dict of sets | `{"ana": {"ben", "cy"}}` | "who is connected to whom", no duplicates |

### Choosing between them

The same gradebook, three ways:

In [ ]:
# A: list of dicts. Easy to loop over, slow to find one student.
a = [{"name": "Ana", "math": 90}, {"name": "Ben", "math": 72}]

# B: dict of dicts. One lookup to find a student.
b = {"Ana": {"math": 90}, "Ben": {"math": 72}}

# C: dict of lists. Grades only; the name is the key.
c = {"Ana": [90, 85], "Ben": [72]}

print(b["Ben"]["math"])     # direct
for row in a:               # have to search
    if row["name"] == "Ben":
        print(row["math"])

There is no single right answer: ask *how will I use this most?* If it is "find by name", make the name a key. If it is "go through everything", a list of records is natural.

### A warning: copies of nested data

`.copy()` copies only the **outer** collection. The inner lists or dicts are still shared. ⚠️ Not an error, but a trap:

In [ ]:
original = {"ana": [90, 85]}
shallow = original.copy()
shallow["ana"].append(100)          # changes the inner list...
print(original)                     # ...which BOTH dicts share

To get a fully independent nested copy you need more than `.copy()`; a tool for that appears later in the course. For now, remember: with nested data, assume the inner parts are shared.

### Try it

Build `team = {"Reds": ["Ana", "Ben"], "Blues": ["Cy"]}`. Add `"Dee"` to the Blues, then print how many players are on each team, one line per team.

In [ ]:
# your code here

## 7 · Common mistakes

### 1. Using a list as a dictionary key

If you want a coordinate pair as a key, use a tuple, not a list. ⚠️ **Expected to fail.**

In [ ]:
grid = {}
grid[[1, 2]] = "tree"

In [ ]:
grid = {}
grid[(1, 2)] = "tree"
print(grid)

### 2. Indexing a set

Sets have no positions, so there is no "first item". ⚠️ **Expected to fail.**

In [ ]:
letters = {"a", "b", "c"}
print(letters[0])

If you need an order, make a list: `sorted(letters)[0]`.

### 3. Expecting a set to keep duplicates or order

In [ ]:
scores = [3, 1, 3, 2, 3]
print(len(scores), len(set(scores)))   # 5 items, but only 3 distinct

If you need to *count* repeats, a set is the wrong tool; that is a dict's job (notebook 14's counter pattern).

### 4. `{}` is an empty dict, not an empty set

In [ ]:
empty_a = {}
empty_b = set()
print(type(empty_a), type(empty_b))

### 5. Keeping records in a list and searching them over and over

Not an error, just a design smell. If the main question is "find the record for *this name*", a dict is the better tool:

In [ ]:
phone_list = [("Ana", "555-1"), ("Ben", "555-2"), ("Cy", "555-3")]
phone_dict = {"Ana": "555-1", "Ben": "555-2", "Cy": "555-3"}

found = None
for name, number in phone_list:     # visit entries until we find it
    if name == "Cy":
        found = number
print(found, phone_dict["Cy"])      # one step

### 6. Sharing one inner list by accident

Writing the inner list once and reusing it gives every key the *same* list:

In [ ]:
shared = []
bad = {"a": shared, "b": shared}
bad["a"].append(1)
print(bad)                          # "b" changed too!

good = {"a": [], "b": []}           # a separate list for each key
good["a"].append(1)
print(good)

### 7. Forgetting what a dict loop gives you

`for x in some_dict` gives **keys**. If you want values or pairs, say so with `.values()` or `.items()`.

In [ ]:
stock = {"apples": 4, "pears": 7}
for thing in stock:
    print(thing)                    # keys only
for thing, count in stock.items():
    print(thing, count)

## 8 · Exercises

The exercises start with choosing a structure and end with modelling a small gradebook. Only use what you have seen in notebooks 12 to 16.

### Exercise 16.1 · Warm-up: pick the structure

For each scenario, set the variable to the **name of the best structure**, as a string: `"list"`, `"tuple"`, `"dict"` or `"set"`.

- `scenario_a`: the seven days of the week, which never change and have a fixed order
- `scenario_b`: phone numbers you look up by person's name
- `scenario_c`: the IDs of visitors seen today, where you only ask "have I seen this ID?"
- `scenario_d`: a to-do list that you add to and tick items off, in order

In [ ]:
scenario_a = None
scenario_b = None
scenario_c = None
scenario_d = None

In [ ]:
check(scenario_a, "tuple")
check(scenario_b, "dict")
check(scenario_c, "set")
check(scenario_d, "list")

### Exercise 16.2 · Warm-up: which can be a key?

Here are five values:

- (a) `"apple"`
- (b) `[1, 2]`
- (c) `(1, 2)`
- (d) `3.5`
- (e) `{1, 2}`

Set `valid_keys` to a string of the **letters** of those that can be used as dictionary keys, in alphabetical order (for example `"abc"`).

In [ ]:
valid_keys = None

In [ ]:
check(valid_keys, "acd")

### Exercise 16.3 · Warm-up: distinct and sorted

Given the list `words`, set `unique_sorted` to a **list** of the distinct words in alphabetical order, and `n_unique` to how many distinct words there are.

In [ ]:
words = ["pear", "apple", "pear", "fig", "apple"]

unique_sorted = None
n_unique = None

In [ ]:
check(unique_sorted, ["apple", "fig", "pear"])
check(n_unique, 3)

### Exercise 16.4 · Practice: a board with tuple keys

`moves` is a list of `(row, col, mark)` tuples. Build a dict `board` whose **keys are `(row, col)` tuples** and whose values are the marks. Loop over `moves` and unpack each tuple.

In [ ]:
moves = [(0, 0, "X"), (1, 1, "O"), (0, 2, "X")]

board = {}
# your loop here

In [ ]:
check(board, {(0, 0): "X", (1, 1): "O", (0, 2): "X"})
check((1, 1) in board, True)

### Exercise 16.5 · Practice: list of dicts: stock value

`products` is a list of dicts. Compute `stock_value` (the sum of `price * stock` over all products, a float) and `in_stock`, a list of the **names** of products whose `stock` is greater than 0 (in the original order).

In [ ]:
products = [
    {"name": "pen", "price": 1.5, "stock": 10},
    {"name": "book", "price": 12.0, "stock": 3},
    {"name": "bag", "price": 25.0, "stock": 0},
]

stock_value = 0.0
in_stock = []
# your loop here

In [ ]:
check(stock_value, 51.0)
check(in_stock, ["pen", "book"])

### Exercise 16.6 · Practice: from parallel lists to records

Two parallel lists hold names and scores. Build `students`, a **list of dicts** like `{"name": "Ana", "score": 90}`, one per person, in the same order. (`zip` from notebook 10 pairs them up.)

In [ ]:
names = ["Ana", "Ben", "Cy"]
scores = [90, 72, 85]

students = []
# your loop here

In [ ]:
check(students, [
    {"name": "Ana", "score": 90},
    {"name": "Ben", "score": 72},
    {"name": "Cy", "score": 85},
])

### Exercise 16.7 · Practice: tags: the right tool for overlap

Two blog posts have these tags. Set `shared` to a **sorted list** of tags on both posts, and `only_a` to a **sorted list** of tags that are only on post A.

In [ ]:
post_a = {"python", "code", "beginner", "tips"}
post_b = {"tips", "python", "career"}

shared = None
only_a = None

In [ ]:
check(shared, ["python", "tips"])
check(only_a, ["beginner", "code"])

### Exercise 16.8 · Stretch: gradebook, part 1: build it

Model a small gradebook. `records` is a list of `(student, subject, score)` tuples. Build `gradebook`, a **dict of dicts**: the outer keys are student names, and each value is a dict from subject to score. For example `gradebook["Ana"]["math"]` is `90`.

Hint: when you meet a student for the first time you need to create their inner dict (`setdefault` or an `in` test will do).

In [ ]:
records = [
    ("Ana", "math", 90),
    ("Ana", "art", 80),
    ("Ben", "math", 70),
    ("Ben", "art", 65),
    ("Cy", "math", 85),
    ("Cy", "art", 95),
    ("Ben", "music", 75),
]

gradebook = {}
# your loop here

In [ ]:
check(gradebook, {
    "Ana": {"math": 90, "art": 80},
    "Ben": {"math": 70, "art": 65, "music": 75},
    "Cy": {"math": 85, "art": 95},
})

### Exercise 16.9 · Stretch: gradebook, part 2: report

Using the finished `gradebook` below, build:

- `averages`: a dict from each student's name to their average score, rounded to 1 decimal place (use `round(x, 1)`)
- `best_student`: the name of the student with the highest average
- `needs_help`: a sorted list of the names of students whose average is below 75

In [ ]:
gradebook = {
    "Ana": {"math": 90, "art": 80},
    "Ben": {"math": 70, "art": 65, "music": 75},
    "Cy": {"math": 85, "art": 95},
}

averages = {}
best_student = None
needs_help = []
# your code here

In [ ]:
check(averages, {"Ana": 85.0, "Ben": 70.0, "Cy": 90.0})
check(best_student, "Cy")
check(needs_help, ["Ben"])

### Exercise 16.10 · Stretch: an index of words to lines

Build `index`, a dict that maps each word to the **set of line numbers** (starting at 1) on which it appears. Then set `cat_lines` to the sorted list of line numbers for `"cat"`.

Think about which structure each level needs: a dict to look a word up, and a set so that a word repeated on one line is only recorded once.

In [ ]:
lines = ["the cat sat", "the dog sat", "a cat ran"]

index = {}
cat_lines = None
# your code here

In [ ]:
check(index, {
    "the": {1, 2}, "cat": {1, 3}, "sat": {1, 2},
    "dog": {2}, "a": {3}, "ran": {3},
})
check(cat_lines, [1, 3])

## 9 · Recap and what's next

- **List**: ordered, mutable, duplicates allowed. The default for "a sequence that changes".
- **Tuple**: ordered, immutable. For fixed bundles, coordinates, and composite keys.
- **Dict**: key → value lookup, mutable. For anything you find by a label.
- **Set**: unique items, no order, mutable. For membership, deduplication and overlaps.
- **String**: ordered, immutable text.
- **Mutable** (list, dict, set) can be changed in place and can be aliased. **Immutable** (tuple, str, numbers) cannot.
- Only **immutable** values can be dict keys or set items; a list cannot, a tuple of immutables can.
- Collections nest: **list of dicts** (records), **dict of lists** (groups), **dict of dicts** (records by name). Pick the shape for the way you will *use* the data most. Copies of nested data share their inner parts.

**Next up:** [17_Functions](17_Functions.ipynb), the second half of Level 2. You have the data structures; next you learn to package code into reusable, named, testable pieces.